# Bracket Simulation — Per-Round Models (Local)

Trains a separate XGBClassifier, XGBRegressor (spread), and XGBRegressor (total) for each tournament round (R1–R6) — 18 models total.  
Simulates the full 2026 bracket: Play-In → R64 → R32 → Sweet 16 → Elite 8 → Final Four → Championship.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor

warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
final = (
    pd.read_csv('data_2026/final_features.csv')
    .rename(columns={
        'WTEAMID': 'W_TEAMID',
        'LTEAMID': 'L_TEAMID',
        'WSCORE':  'W_SCORE',
        'LSCORE':  'L_SCORE',
    })
)
final['WIN_INDICATOR'] = 1

drop_cols = ['W_CTWINS', 'W_AVERAGECTSCORE', 'L_CTWINS', 'L_AVERAGECTSCORE',
             'W_WLOCN', 'W_WLOCH', 'W_WLOCA', 'L_WLOCN', 'L_WLOCH', 'L_WLOCA']
final = final.drop(columns=[c for c in drop_cols if c in final.columns])

NON_FEATURE_COLS = {'SEASON', 'WIN_INDICATOR', 'L_TEAMID', 'W_TEAMID',
                    'W_SCORE', 'L_SCORE', 'ROUND', 'L_REGION', 'W_REGION'}
feature_cols = [c for c in final.columns if c not in NON_FEATURE_COLS]

BRACKET_SEASON = 2026

train = final[(final.SEASON >= 2010) & (final.SEASON <= 2023)].copy()
test  = final[final.SEASON > 2023].copy()

print(f'Train seasons: {sorted(train.SEASON.unique())}')
print(f'Test seasons:  {sorted(test.SEASON.unique())}')
print(f'Feature columns: {len(feature_cols)}')

## Augment training data (swap W/L orientations)

In [ ]:
def swap_wl(df):
    """Return a copy of df with all W_ and L_ columns swapped."""
    out = df.copy()
    w_cols = sorted([c for c in df.columns if c.startswith('W_')])
    l_cols = ['L_' + c[2:] for c in w_cols]
    for w, l in zip(w_cols, l_cols):
        out[w] = df[l]
        out[l] = df[w]
    return out


df_swapped = swap_wl(train)
train_aug = pd.concat([train, df_swapped], ignore_index=True)
train_aug['WIN_INDICATOR'] = (train_aug['W_SCORE'] > train_aug['L_SCORE']).astype(int)

print(f'Augmented training rows: {len(train_aug)}')

## Train per-round models — win, spread, total (R1–R6)

ROUND values: 1=R64, 2=R32, 3=S16, 4=E8, 5=FF, 6=Championship  
**18 models total**: 6 XGBClassifier (win) + 6 XGBRegressor (spread) + 6 XGBRegressor (total)

In [ ]:
# Fixed best params (from GridSearchCV in 3_scores_2026.ipynb)
WIN_PARAMS    = dict(learning_rate=0.1, max_depth=4, min_child_weight=4, n_estimators=100)
SPREAD_PARAMS = dict(learning_rate=0.1, max_depth=3, min_child_weight=2, n_estimators=100)
TOTAL_PARAMS  = dict(learning_rate=0.1, max_depth=3, min_child_weight=2, n_estimators=100)

round_models  = {}
spread_models = {}
total_models  = {}
round_names   = {1: 'R64', 2: 'R32', 3: 'S16', 4: 'E8', 5: 'FF', 6: 'Championship'}

for r in range(1, 7):
    round_train = train_aug[train_aug.ROUND == r]
    if len(round_train) == 0:
        print(f'Round {r} ({round_names[r]}): no training data — skipping')
        continue

    X = round_train[feature_cols]
    y_win    = round_train['WIN_INDICATOR']
    y_spread = round_train['W_SCORE'] - round_train['L_SCORE']
    y_total  = round_train['W_SCORE'] + round_train['L_SCORE']

    win_model = XGBClassifier(eval_metric='logloss', **WIN_PARAMS)
    win_model.fit(X, y_win)
    round_models[r] = win_model

    spr_model = XGBRegressor(eval_metric='rmse', **SPREAD_PARAMS)
    spr_model.fit(X, y_spread)
    spread_models[r] = spr_model

    tot_model = XGBRegressor(eval_metric='rmse', **TOTAL_PARAMS)
    tot_model.fit(X, y_total)
    total_models[r] = tot_model

    print(f'Round {r} ({round_names[r]}): trained on {len(round_train)} rows')

print(f'\n18 models trained across {len(round_models)} rounds (win + spread + total each)')

## Per-round accuracy on 2024–2025 test data

In [ ]:
rows = []
for r in sorted(round_models.keys()):
    test_r = test[test.ROUND == r].copy()
    if len(test_r) == 0:
        continue
    test_r_aug = pd.concat([test_r, swap_wl(test_r)], ignore_index=True)
    test_r_aug['WIN_INDICATOR'] = (test_r_aug['W_SCORE'] > test_r_aug['L_SCORE']).astype(int)
    test_r_aug['ACTUAL_SPREAD'] = test_r_aug['W_SCORE'] - test_r_aug['L_SCORE']
    test_r_aug['ACTUAL_TOTAL']  = test_r_aug['W_SCORE'] + test_r_aug['L_SCORE']

    win_preds    = round_models[r].predict(test_r_aug[feature_cols])
    spread_preds = spread_models[r].predict(test_r_aug[feature_cols])
    total_preds  = total_models[r].predict(test_r_aug[feature_cols])

    rows.append({
        'Round':      r,
        'Name':       round_names[r],
        'N_Games':    len(test_r),
        'Win_Acc':    round(accuracy_score(test_r_aug['WIN_INDICATOR'], win_preds), 4),
        'Spread_MAE': round(mean_absolute_error(test_r_aug['ACTUAL_SPREAD'], spread_preds), 2),
        'Total_MAE':  round(mean_absolute_error(test_r_aug['ACTUAL_TOTAL'],  total_preds), 2),
    })

acc_df = pd.DataFrame(rows)
print('Per-round model evaluation on 2024–2025 test data:')
display(acc_df)

## Helper functions

In [ ]:
def predict_games(win_model, spread_model, total_model, df, feature_cols):
    """Run all three model predictions, filling missing feature cols with 0."""
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    df['PRED_WIN_INDICATOR'] = win_model.predict(df[feature_cols])
    df['PRED_SPREAD'] = spread_model.predict(df[feature_cols]).round(1)
    df['PRED_TOTAL']  = total_model.predict(df[feature_cols]).round(1)
    return df


def add_team_names(result, teams):
    """Join team names onto a result dataframe that has W_TEAMID and L_TEAMID."""
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='L_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'L_TEAM_NAME'}).drop(columns=['TEAMID'])
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='W_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'W_TEAM_NAME'}).drop(columns=['TEAMID'])
    return result


def get_round_winners(result):
    """Pick the predicted winner from each game."""
    r = result.copy()
    r['W_TEAM_ID']   = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAMID,    r.L_TEAMID)
    r['W_TEAM_NAME'] = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAM_NAME, r.L_TEAM_NAME)
    r['W_SEED']      = np.where(r.PRED_WIN_INDICATOR == 1, r.W_SEED,      r.L_SEED)
    r['W_REGION']    = np.where(r.PRED_WIN_INDICATOR == 1, r.W_REGION,    r.L_REGION)
    return r[['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION']]


def add_season_stats(matchups, season_stats, season_year):
    """Join season stats onto matchups that have WTEAMID2 and LTEAMID2 columns."""
    season = season_stats[season_stats.SEASON == season_year].drop(columns=['SEASON'])
    season = season.drop(columns=[c for c in ['REGION'] if c in season.columns])
    season_w = season.copy()
    season_w.columns = ['W_' + c for c in season_w.columns]
    season_l = season.copy()
    season_l.columns = ['L_' + c for c in season_l.columns]

    df = matchups.merge(season_w, left_on='WTEAMID2', right_on='W_TEAMID').drop(columns=['W_TEAMID'])
    df = df.merge(season_l, left_on='LTEAMID2', right_on='L_TEAMID').drop(columns=['L_TEAMID'])
    df = df.rename(columns={'WTEAMID2': 'W_TEAMID', 'LTEAMID2': 'L_TEAMID'})
    return df

## Load 2026 bracket data

In [ ]:
teams = pd.read_csv('data_2026/MTeams.csv')
teams.columns = teams.columns.str.upper()

season_stats = pd.read_csv('data_2026/final_season_stats.csv')

seeds_raw = pd.read_csv('data_2026/MNCAATourneySeeds.csv')
seeds_raw.columns = seeds_raw.columns.str.upper()
seeds_bracket = seeds_raw[seeds_raw.SEASON == BRACKET_SEASON].copy()
seeds_bracket['REGION']    = seeds_bracket['SEED'].str[0]
seeds_bracket['IS_PLAYIN'] = seeds_bracket['SEED'].str[1:].str.contains('[ab]', regex=True)
seeds_bracket['SEED_NUM']  = seeds_bracket['SEED'].str[1:].str.replace('[a-z]', '', regex=True).astype(int)

playin_seeds  = seeds_bracket[seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})
regular_seeds = seeds_bracket[~seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})

print(f'{BRACKET_SEASON} seeds loaded: {len(seeds_bracket)} total, {len(playin_seeds)} play-in, {len(regular_seeds)} regular')

## Play-In Games
Uses `round_models[1]`, `spread_models[1]`, `total_models[1]` (R64 round models).

In [ ]:
playin_rows = []
for (region, seed), group in playin_seeds.groupby(['REGION', 'SEED']):
    team_ids = group.TEAMID.tolist()
    if len(team_ids) == 2:
        playin_rows.append({
            'WTEAMID2': team_ids[0], 'LTEAMID2': team_ids[1],
            'W_SEED': seed, 'L_SEED': seed,
            'W_REGION': region, 'L_REGION': region,
        })

if playin_rows:
    playin_matchups = pd.DataFrame(playin_rows)
    games_playin    = add_season_stats(playin_matchups, season_stats, BRACKET_SEASON)
    result_playin   = predict_games(round_models[1], spread_models[1], total_models[1], games_playin, feature_cols)
    result_playin   = add_team_names(result_playin, teams)
    playin_result   = result_playin[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED',
                                     'L_REGION', 'W_TEAM_NAME', 'W_REGION',
                                     'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
    playin_winners  = get_round_winners(playin_result)
    print(f'Play-In ({BRACKET_SEASON}):')
    display(playin_result[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'W_REGION',
                            'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']].sort_values('W_REGION'))
else:
    playin_winners = pd.DataFrame(columns=['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION'])
    print(f'No play-in games found for {BRACKET_SEASON}')

## Round of 64 — R1 model

In [ ]:
ROUND_1_PAIRS = [(1, 16), (8, 9), (5, 12), (4, 13), (6, 11), (3, 14), (7, 10), (2, 15)]

round1_seeds = regular_seeds.copy()
if not playin_winners.empty:
    playin_w_seeds = playin_winners.rename(
        columns={'W_TEAM_ID': 'TEAMID', 'W_SEED': 'SEED', 'W_REGION': 'REGION'}
    )[['TEAMID', 'REGION', 'SEED']]
    round1_seeds = pd.concat([round1_seeds, playin_w_seeds], ignore_index=True)

round1_rows = []
for region in round1_seeds.REGION.unique():
    r = round1_seeds[round1_seeds.REGION == region].set_index('SEED')
    for s1, s2 in ROUND_1_PAIRS:
        if s1 in r.index and s2 in r.index:
            round1_rows.append({
                'WTEAMID2': r.loc[s1, 'TEAMID'], 'LTEAMID2': r.loc[s2, 'TEAMID'],
                'W_SEED': s1, 'L_SEED': s2,
                'W_REGION': region, 'L_REGION': region,
            })

round1_matchups = pd.DataFrame(round1_rows)
games_r1 = add_season_stats(round1_matchups, season_stats, BRACKET_SEASON)
result   = predict_games(round_models[1], spread_models[1], total_models[1], games_r1, feature_cols)
result   = add_team_names(result, teams)

round_1_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED',
                           'L_REGION', 'W_TEAM_NAME', 'W_REGION',
                           'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
print(f'Round of 64 — {len(round_1_results)} games')
display(round_1_results[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'L_SEED', 'W_REGION',
                          'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']].sort_values('W_REGION'))

In [ ]:
round_1_winners = get_round_winners(round_1_results)

## Round of 32 — R2 model

In [ ]:
ROUND_2_SEED_PAIRS = [
    (1, 8), (1, 9), (16, 8), (16, 9),
    (4, 5), (4, 12), (13, 5), (13, 12),
    (3, 6), (3, 11), (14, 6), (14, 11),
    (2, 7), (2, 10), (15, 7), (15, 10),
]

df1  = round_1_winners.rename(columns={'W_TEAM_ID': 'WTEAMID2', 'W_SEED': 'W_SEED', 'W_REGION': 'W_REGION', 'W_TEAM_NAME': 'W_TEAM_NAME'})
df2  = round_1_winners.rename(columns={'W_TEAM_ID': 'LTEAMID2', 'W_SEED': 'L_SEED', 'W_REGION': 'L_REGION', 'W_TEAM_NAME': 'L_TEAM_NAME'})
cross = df1.merge(df2, how='cross', suffixes=('', '_r'))

seed_set = set(ROUND_2_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
second_round_matchups = cross[mask][['W_TEAM_NAME', 'W_REGION', 'WTEAMID2', 'W_SEED',
                                     'L_TEAM_NAME', 'L_REGION', 'LTEAMID2', 'L_SEED']]

games_r2 = add_season_stats(second_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']],
                             season_stats, BRACKET_SEASON)
result   = predict_games(round_models[2], spread_models[2], total_models[2], games_r2, feature_cols)
result   = add_team_names(result, teams)

round_2_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED',
                           'L_REGION', 'W_TEAM_NAME', 'W_REGION',
                           'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
print(f'Round of 32 — {len(round_2_results)} games')
display(round_2_results[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'L_SEED', 'W_REGION',
                          'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']].sort_values('W_REGION'))

In [ ]:
round_2_winners = get_round_winners(round_2_results)

## Sweet 16 — R3 model

In [ ]:
ROUND_3_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1, 16, 8, 9]
    for s2 in [5, 12, 4, 13]
] + [
    (s1, s2)
    for s1 in [6, 11, 3, 14]
    for s2 in [7, 10, 2, 15]
]

df1  = round_2_winners.rename(columns={'W_TEAM_ID': 'WTEAMID2', 'W_SEED': 'W_SEED', 'W_REGION': 'W_REGION', 'W_TEAM_NAME': 'W_TEAM_NAME'})
df2  = round_2_winners.rename(columns={'W_TEAM_ID': 'LTEAMID2', 'W_SEED': 'L_SEED', 'W_REGION': 'L_REGION', 'W_TEAM_NAME': 'L_TEAM_NAME'})
cross = df1.merge(df2, how='cross', suffixes=('', '_r'))

seed_set = set(ROUND_3_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
third_round_matchups = cross[mask][['W_TEAM_NAME', 'W_REGION', 'WTEAMID2', 'W_SEED',
                                    'L_TEAM_NAME', 'L_REGION', 'LTEAMID2', 'L_SEED']]

games_r3 = add_season_stats(third_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']],
                             season_stats, BRACKET_SEASON)
result   = predict_games(round_models[3], spread_models[3], total_models[3], games_r3, feature_cols)
result   = add_team_names(result, teams)

sweet_16_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED',
                            'L_REGION', 'W_TEAM_NAME', 'W_REGION',
                            'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
print(f'Sweet 16 — {len(sweet_16_results)} games')
display(sweet_16_results[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'L_SEED', 'W_REGION',
                           'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']].sort_values('W_REGION'))

In [ ]:
sweet_16_winners = get_round_winners(sweet_16_results)

## Elite 8 — R4 model

In [ ]:
ELITE_8_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1, 16, 8, 9, 5, 12, 4, 13]
    for s2 in [6, 11, 3, 14, 7, 10, 2, 15]
]

df1  = sweet_16_winners.rename(columns={'W_TEAM_ID': 'WTEAMID2', 'W_SEED': 'W_SEED', 'W_REGION': 'W_REGION', 'W_TEAM_NAME': 'W_TEAM_NAME'})
df2  = sweet_16_winners.rename(columns={'W_TEAM_ID': 'LTEAMID2', 'W_SEED': 'L_SEED', 'W_REGION': 'L_REGION', 'W_TEAM_NAME': 'L_TEAM_NAME'})
cross = df1.merge(df2, how='cross', suffixes=('', '_r'))

seed_set = set(ELITE_8_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
elite_8_matchups = cross[mask][['W_TEAM_NAME', 'W_REGION', 'WTEAMID2', 'W_SEED',
                                'L_TEAM_NAME', 'L_REGION', 'LTEAMID2', 'L_SEED']]

games_r4 = add_season_stats(elite_8_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']],
                             season_stats, BRACKET_SEASON)
result   = predict_games(round_models[4], spread_models[4], total_models[4], games_r4, feature_cols)
result   = add_team_names(result, teams)

elite_8_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED',
                           'L_REGION', 'W_TEAM_NAME', 'W_REGION',
                           'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
print(f'Elite 8 — {len(elite_8_results)} games')
display(elite_8_results[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'L_SEED', 'W_REGION',
                          'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']].sort_values('W_REGION'))

In [ ]:
elite_8_winners = get_round_winners(elite_8_results)

## Final Four — R5 model

In [ ]:
# W vs X regions, Y vs Z regions
wx = elite_8_winners[elite_8_winners.W_REGION == 'W'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'X'].rename(
        columns={'W_TEAM_ID': 'TEAM_2', 'W_TEAM_NAME': 'W_TEAM_NAME_2', 'W_SEED': 'SEED_2', 'W_REGION': 'REGION_2'}
    ),
    how='cross',
)
yz = elite_8_winners[elite_8_winners.W_REGION == 'Y'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'Z'].rename(
        columns={'W_TEAM_ID': 'TEAM_2', 'W_TEAM_NAME': 'W_TEAM_NAME_2', 'W_SEED': 'SEED_2', 'W_REGION': 'REGION_2'}
    ),
    how='cross',
)

final_four_matchups = pd.concat([wx, yz]).rename(columns={
    'W_TEAM_ID':    'WTEAMID2',
    'W_TEAM_NAME_2':'L_TEAM_NAME',
    'REGION_2':     'L_REGION',
    'TEAM_2':       'LTEAMID2',
    'SEED_2':       'L_SEED',
})[['W_TEAM_NAME', 'W_REGION', 'WTEAMID2', 'W_SEED', 'L_TEAM_NAME', 'L_REGION', 'LTEAMID2', 'L_SEED']]

games_r5 = add_season_stats(final_four_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']],
                             season_stats, BRACKET_SEASON)
result   = predict_games(round_models[5], spread_models[5], total_models[5], games_r5, feature_cols)
result   = add_team_names(result, teams)

final_four_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED',
                              'L_REGION', 'W_TEAM_NAME', 'W_REGION',
                              'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
print(f'Final Four — {len(final_four_results)} games')
display(final_four_results[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'L_SEED', 'W_REGION',
                             'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']])

In [ ]:
final_four_winners = get_round_winners(final_four_results)

## Championship — R6 model

In [ ]:
championship_matchup = (
    final_four_winners.rename(columns={'W_TEAM_ID': 'WTEAMID2', 'W_TEAM_NAME': 'W_TEAM_NAME', 'W_SEED': 'W_SEED', 'W_REGION': 'W_REGION'})
    .merge(
        final_four_winners.rename(columns={'W_TEAM_ID': 'LTEAMID2', 'W_TEAM_NAME': 'L_TEAM_NAME', 'W_SEED': 'L_SEED', 'W_REGION': 'L_REGION'}),
        how='cross',
    )
    .query('WTEAMID2 != LTEAMID2')
    .head(1)
)

games_r6 = add_season_stats(championship_matchup[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']],
                             season_stats, BRACKET_SEASON)
result   = predict_games(round_models[6], spread_models[6], total_models[6], games_r6, feature_cols)
result   = add_team_names(result, teams)

championship_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED',
                                'L_REGION', 'W_TEAM_NAME', 'W_REGION',
                                'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']]
print('Championship:')
display(championship_results[['W_TEAM_NAME', 'L_TEAM_NAME', 'W_SEED', 'L_SEED',
                               'PRED_WIN_INDICATOR', 'PRED_SPREAD', 'PRED_TOTAL']])

## Champion

In [ ]:
champion = get_round_winners(championship_results)
c = champion.iloc[0]
print(f'2026 Predicted Champion: {c.W_TEAM_NAME} (Seed {int(c.W_SEED)}, Region {c.W_REGION})')